In [ ]:
# parameters
dir1 = r"D:\Data\HDF5DB_Test"
dir2 = r"D:\Data\HDF5DB"
target_table = "stock_cn_status"
target_factor = "if_listed"

# 参数说明

| 参数 | 含义 |
|------|------|
| `dir1` | 第一个因子库主目录 |
| `dir2` | 第二个因子库主目录 |
| `target_table` | 待分析的表名 |
| `target_factor` | 待分析的因子名 |

In [2]:
# 全局设置
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False# 正确显示负号
import matplotlib.pyplot as plt
from IPython.display import HTML

from QuantStudio.Factor.HDF5DB import HDF5DB

FDB1 = HDF5DB(args={"MainDir": dir1}).connect()
FDB2 = HDF5DB(args={"MainDir": dir2}).connect()

In [3]:
# 读取两个库中目标因子的数据
# 注意: readData 需要显式传入 ids/dts, 这里直接读取全量数据后对齐
FT1 = FDB1.getTable(target_table)
FT2 = FDB2.getTable(target_table)

# 先取两个库各自完整的 ID / DateTime 索引
ids1, dts1 = FT1.getID(target_factor), FT1.getDateTime(target_factor)
ids2, dts2 = FT2.getID(target_factor), FT2.getDateTime(target_factor)

Data1 = FT1.readFactorData(target_factor, ids=ids1, dts=dts1)
Data2 = FT2.readFactorData(target_factor, ids=ids2, dts=dts2)

print("库1 数据形状:", Data1.shape, "| 库2 数据形状:", Data2.shape)
print("库1 ID 数:", len(ids1), "| 库2 ID 数:", len(ids2))
print("库1 时点数:", len(dts1), "| 库2 时点数:", len(dts2))

库1 数据形状: (3043, 5865) | 库2 数据形状: (3043, 5865)
库1 ID 数: 5865 | 库2 ID 数: 5865
库1 时点数: 3043 | 库2 时点数: 3043


In [4]:
# 索引集合差异分析
ids_only1 = sorted(set(ids1) - set(ids2))
ids_only2 = sorted(set(ids2) - set(ids1))
dts_only1 = sorted(set(dts1) - set(dts2))
dts_only2 = sorted(set(dts2) - set(dts1))

print(f"仅库1拥有的 ID 数: {len(ids_only1)}  仅库2拥有的 ID 数: {len(ids_only2)}")
print(f"仅库1拥有的时点数: {len(dts_only1)}  仅库2拥有的时点数: {len(dts_only2)}")

# 交集索引对齐
IDs = sorted(set(ids1) & set(ids2))
DTs = sorted(set(dts1) & set(dts2))
df1 = Data1.reindex(index=DTs, columns=IDs)
df2 = Data2.reindex(index=DTs, columns=IDs)

print(f"\n交集部分形状: {df1.shape}  (时点 x ID = {len(DTs)} x {len(IDs)})")

仅库1拥有的 ID 数: 0  仅库2拥有的 ID 数: 0
仅库1拥有的时点数: 0  仅库2拥有的时点数: 0

交集部分形状: (3043, 5865)  (时点 x ID = 3043 x 5865)


In [5]:
# 差异定位: 找出两个 DataFrame 中值不一致的位置
is_num1 = pd.api.types.is_numeric_dtype(df1.dtypes)
is_num2 = pd.api.types.is_numeric_dtype(df2.dtypes)

if is_num1 and is_num2:
    # 数值型: 直接做差取绝对值
    diff = (df1 - df2).abs()
else:
    # 非数值型(如字符串): 用逐元素不相等判断
    diff = (df1 != df2).astype(float)

# 缺失值处理: NaN 与 NaN 视为一致, 一个 NaN 一个非 NaN 视为差异
equal_mask = (df1 == df2) | (df1.isna() & df2.isna())
n_total = df1.size
n_diff = int((~equal_mask).sum().sum())

print(f"总元素数: {n_total}")
print(f"不一致元素数: {n_diff}  ({n_diff / n_total:.4%})")

if n_diff == 0:
    print("两库目标因子在交集范围内完全一致。")
else:
    # 数值型提供差异幅度统计
    if is_num1 and is_num2:
        diff_flat = diff.values[~np.isnan(diff.values)]
        print("\n--- 差异幅度统计 (仅非零差异) ---")
        print("差异均值:", diff_flat.mean())
        print("差异最大值:", diff_flat.max())
        print("差异中位数:", np.median(diff_flat))

    # 按时间维度聚合差异
    diff_by_dt = (~equal_mask).sum(axis=1)
    top_dts = diff_by_dt[diff_by_dt > 0].sort_values(ascending=False).head(10)
    print("\n--- 差异最多的前 10 个时点 ---")
    for dt_val, cnt in top_dts.items():
        print(f"  {dt_val}: {cnt} 个 ID 不一致")

    # 按 ID 维度聚合差异
    diff_by_id = (~equal_mask).sum(axis=0)
    top_ids = diff_by_id[diff_by_id > 0].sort_values(ascending=False).head(10)
    print("\n--- 差异最多的前 10 个 ID ---")
    for id_val, cnt in top_ids.items():
        print(f"  {id_val}: {cnt} 个时点不一致")

    # 展示差异样本 (定位若干具体位置)
    samp_count = min(10, n_diff)
    diff_positions = list(zip(*np.where(~equal_mask.values)))[:samp_count]
    print(f"\n--- 差异位置样本 (前 {samp_count} 个) ---")
    for r, c in diff_positions:
        dt_val = DTs[r]
        id_val = IDs[c]
        print(f"  [{dt_val}][{id_val}]  库1={df1.iat[r, c]}  库2={df2.iat[r, c]}")

总元素数: 17847195
不一致元素数: 0  (0.0000%)
两库目标因子在交集范围内完全一致。
